In [ ]:
import os
import json
import csv
import zipfile
import random
from collections import defaultdict, Counter

# =========================
# 0) 설정
# =========================
ZIP_DIR   = "../data/raw/clickbait_labels"   # zip 42개가 있는 폴더
OUT_CSV   = "../data/impact/impact2_P2_subjectN_500_balanced.csv"
OUT_JSON  = "../data/impact/impact2_P2_subjectN_500_balanced.json"

TARGET_N = 500

# 낚시성(useType=0)만 유지하려면 True (이전 요구 흐름상 기본 True 권장)
ONLY_USETYPE_0 = True

# 너무 짧은 N문장 방지(필요 시 15~25로 상향)
MIN_SENT_LEN = 10

# 랜덤 시드(재현성)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# =========================
# 1) 유틸
# =========================
def s(x):
    return "" if x is None else str(x).strip()

def first_n_sentence(process_sentence_info):
    """
    processSentenceInfo 리스트에서 subjectConsistencyYn == 'N'인 첫 문장 1개를 반환
    """
    if not isinstance(process_sentence_info, list):
        return None
    for item in process_sentence_info:
        if s(item.get("subjectConsistencyYn")).upper() == "N":
            sent = s(item.get("sentenceContent"))
            if len(sent) >= MIN_SENT_LEN:
                return sent
    # fallback: 길이 조건 때문에 못 찾았으면, 길이 조건 없이라도 첫 N를 반환
    for item in process_sentence_info:
        if s(item.get("subjectConsistencyYn")).upper() == "N":
            sent = s(item.get("sentenceContent"))
            if sent:
                return sent
    return None

def get_block(obj, keys):
    """
    흔히 쓰는 블록들(sourceDataInfo / labeledDataInfo / labelingDataInfo)을 안정적으로 가져오기
    """
    for k in keys:
        v = obj.get(k)
        if isinstance(v, dict):
            return v
    return {}

# =========================
# 2) ZIP 내부 JSON 전수 순회 + P2 후보 수집
# =========================
zip_files = [
    os.path.join(ZIP_DIR, f) for f in os.listdir(ZIP_DIR)
    if f.lower().endswith(".zip")
]

candidates = []
stats = Counter()

for zpath in zip_files:
    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            # zip 내부 json 파일만 처리
            members = [m for m in zf.namelist() if m.lower().endswith(".json")]
            for m in members:
                try:
                    with zf.open(m) as fp:
                        obj = json.load(fp)
                except Exception:
                    stats["json_load_fail"] += 1
                    continue

                sdi = get_block(obj, ["sourceDataInfo", "dataInfo"])
                ldi = get_block(obj, ["labeledDataInfo", "labelingDataInfo"])

                partNum = s(sdi.get("partNum")).upper()
                if partNum != "P2":
                    stats["skip_not_P2"] += 1
                    continue

                useType = s(sdi.get("useType"))
                if ONLY_USETYPE_0 and useType != "0":
                    stats["skip_useType_not_0"] += 1
                    continue

                newsTitle = s(sdi.get("newsTitle") or sdi.get("title"))
                if not newsTitle:
                    stats["skip_empty_title"] += 1
                    continue

                psi = ldi.get("processSentenceInfo") or []
                one_line = first_n_sentence(psi)
                if not one_line:
                    stats["skip_no_N_sentence"] += 1
                    continue

                row = {
                    "newsCategory": s(sdi.get("newsCategory")),
                    "newsSubcategory": s(sdi.get("newsSubcategory")),
                    "newsTitle": newsTitle,
                    # 핵심: N문장 1줄만 newsContent로
                    "newsContent": one_line,
                    "useType": 0  # 항상 0으로 저장(요구사항)
                }

                # 카테고리 공백이면 제외할지 여부:
                # 균등 분배가 목표이므로 카테고리 없는 것은 제외(권장)
                if not row["newsCategory"]:
                    stats["skip_empty_category"] += 1
                    continue

                candidates.append(row)
                stats["kept_candidates"] += 1

    except Exception:
        stats["zip_open_fail"] += 1
        continue

print("ZIP files:", len(zip_files))
print("Candidates collected:", len(candidates))
print("Top stats:", stats.most_common(12))

# =========================
# 3) 중복 제거: (newsTitle, newsContent)
# =========================
seen = set()
deduped = []
for r in candidates:
    key = (r["newsTitle"], r["newsContent"])
    if key in seen:
        continue
    seen.add(key)
    deduped.append(r)

print("After dedup:", len(deduped))

# =========================
# 4) newsCategory 균등 분배로 500개 고정
# =========================
by_cat = defaultdict(list)
for r in deduped:
    by_cat[r["newsCategory"]].append(r)

cats = list(by_cat.keys())
K = len(cats)
if K == 0:
    raise RuntimeError("newsCategory 기준 후보가 없습니다. (필터/경로/키명 점검 필요)")

# 각 카테고리 셔플(재현성 seed 적용됨)
for c in cats:
    random.shuffle(by_cat[c])

base = TARGET_N // K
remainder = TARGET_N % K

selected = []

# 4-1) base씩 우선 추출
for c in cats:
    take = min(base, len(by_cat[c]))
    selected.extend(by_cat[c][:take])
    by_cat[c] = by_cat[c][take:]

# 4-2) 남는 수량은 후보 많은 카테고리부터 1개씩 라운드로빈
cats_sorted = sorted(cats, key=lambda x: len(by_cat[x]), reverse=True)

while len(selected) < TARGET_N:
    progressed = False
    for c in cats_sorted:
        if len(selected) >= TARGET_N:
            break
        if by_cat[c]:
            selected.append(by_cat[c].pop(0))
            progressed = True
    if not progressed:
        break  # 더 이상 뽑을 후보 없음

if len(selected) < TARGET_N:
    # 500개 고정이 요구사항이라면, 이 경우는 "후보 자체가 부족"입니다.
    raise RuntimeError(f"후보가 부족하여 {TARGET_N}개를 채울 수 없습니다. (현재 {len(selected)}개)")

print("Final selected:", len(selected))
print("Category counts:", Counter([r["newsCategory"] for r in selected]).most_common())

# =========================
# 5) 저장: 요청 5개 필드만
# =========================
fields = ["newsCategory", "newsSubcategory", "newsTitle", "newsContent", "useType"]

with open(OUT_CSV, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in selected:
        w.writerow({k: r.get(k, "") for k in fields})

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump([{k: r.get(k, "") for k in fields} for r in selected], f, ensure_ascii=False, indent=2)

print("Saved:", OUT_CSV, OUT_JSON)
